# v06 · The exact form of what the manifold approximates: (count vector, rank in the anagram class)
Not in the repo; written here. Each string is an arrangement of a multiset; its address is the pair (counts, rank among the distinct arrangements).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
N = 97
chars = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw").HyperWebster._DEFAULT_CHARS
idx = {c: i for i, c in enumerate(chars)}
def rank(text):
    c = Counter(text); syms = sorted(c, key=lambda ch: idx[ch]); m = len(text)
    P = math.factorial(m)
    for k in c.values(): P //= math.factorial(k)
    r = 0
    for ch in text:
        below = sum(c[x] for x in syms if idx[x] < idx[ch])
        r += P * below // m
        P = P * c[ch] // m; c[ch] -= 1; m -= 1
    return r
def unrank(r, counts, L):
    c = dict(counts); syms = sorted(c, key=lambda ch: idx[ch]); m = L
    P = math.factorial(m)
    for k in c.values(): P //= math.factorial(k)
    out = []
    for _ in range(L):
        for x in syms:
            if c[x] == 0: continue
            w = P * c[x] // m
            if r < w: out.append(x); P = w; c[x] -= 1; m -= 1; break
            r -= w
    return "".join(out)
# exhaustive check on a small multiset: every arrangement of 'aabbc' gets a distinct rank 0..29
from itertools import permutations
arr = sorted(set("".join(p) for p in permutations("aabbc")))
ranks = sorted(rank(a) for a in arr)
print(len(arr), "arrangements; ranks are exactly 0..%d:" % (len(arr)-1), ranks == list(range(len(arr))))
assert all(unrank(rank(a), Counter(a), 5) == a for a in arr)

30 arrangements; ranks are exactly 0..29: True


In [3]:
print("     L   naive  log2(multinomial)  +count-vector  total  total/naive  round-trip")
for L in (140, 1000, 4000):
    t = CORPUS[:L]; c = Counter(t); r = rank(t)
    Mn = math.factorial(L)
    for k in c.values(): Mn //= math.factorial(k)
    mult = Mn.bit_length() - 1
    cnt = math.comb(L + N - 1, N - 1).bit_length() - 1        # index of the count vector among all compositions of L into 97 parts
    print("%6d  %6d  %17d  %13d  %5d  %11.3f  %s" % (L, round(L*math.log2(N)), mult, cnt, mult+cnt, (mult+cnt)/(L*math.log2(N)), unrank(r, c, L) == t))

     L   naive  log2(multinomial)  +count-vector  total  total/naive  round-trip
   140     924                564            225    789        0.854  True
  1000    6600               4662            464   5126        0.777  True
  4000   26400              20146            652  20798        0.788  True


Exact, reversible, and the count vector is paid for. The result is the order-0 entropy of the text: it wins because the corpus is not uniform over 97 symbols, not because the addressing is cleverer.